## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


확률적 경사 하강법(SGD), 미니배치 경사 하강법(Mini-batch Gradient Descent)

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

과적합되고 있음을 의미한다. 이 경우, Early Stopping을 진행하거나, regulation을 추가하여 문제를 해결한다.

### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

높은 편향이 문제이다. α를 줄여야 한다.

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

1. 과적합을 방지하기 위해서이다.
2. 불필요한 특성이 많은 경우이기 때문이다.
3. 특성이 많거나 서로 상관된 특성이 많기 때문이다.

### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# 데이터 로드
iris = load_iris()
X = iris.data
y = iris.target

# bias 추가
X = np.c_[np.ones((X.shape[0], 1)), X]

# train / validation 분리
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# one-hot encoding
def one_hot(y, n_classes):
    one_hot = np.zeros((len(y), n_classes))
    one_hot[np.arange(len(y)), y] = 1
    return one_hot

Y_train = one_hot(y_train, 3)
Y_val = one_hot(y_val, 3)

# softmax 함수
def softmax(logits):
    exp = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

# cross entropy loss
def cross_entropy(y_true, y_pred):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-10), axis=1))

# 초기 파라미터
np.random.seed(42)
theta = np.random.randn(X_train.shape[1], 3)

eta = 0.01
n_epochs = 5000

best_loss = np.inf
patience = 10
wait = 0

for epoch in range(n_epochs):

    logits = X_train.dot(theta)
    y_pred = softmax(logits)

    gradients = (1/len(X_train)) * X_train.T.dot(y_pred - Y_train)
    theta -= eta * gradients

    # validation
    val_pred = softmax(X_val.dot(theta))
    val_loss = cross_entropy(Y_val, val_pred)

    if val_loss < best_loss:
        best_loss = val_loss
        best_theta = theta.copy()
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping at epoch:", epoch)
            break

theta = best_theta